# A179320 — Riordan Lie Algebra Generator

This notebook explores the OEIS sequence **A179320** and its role as the infinitesimal generator of the Riordan array **A078812**.

## Setup

The OEIS sequence **A078812** is a triangle read by rows. Arranging its terms into a lower triangular matrix $M$, the entry at row $i$, column $j$ is:

$$M[i][j] = \binom{i+j+1}{2j+1}$$

This is the odd columns of Pascal's triangle, packed into a lower triangular array.

## The matrix logarithm

In the Riordan group Lie algebra, the matrix logarithm $L = \log(M)$ is itself lower triangular, and its entire structure is determined by its first column alone. Specifically, if we write the EGF coefficients of column 0 as $a(n) = A179320(n)$, then every entry of $L$ is given by:

$$L[i][j] = \frac{(j+1)\,a(i-j)}{(i-j)!}$$

So **A179320** is, in a precise sense, the "seed" that generates the full logarithm matrix.

## Recovering the combinatorial structure

As noted in the OEIS entry, exponentiating $L$ recovers $M$:

$$\exp(L) = M, \qquad \exp(L)[i][j] = \binom{i+j+1}{2j+1}$$

In [1]:
import os
from pathlib import Path
from sympy import symbols, pprint, simplify, factor, together

# Change to project root so relative paths in common.py work
os.chdir(Path("__file__").resolve().parent.parent if '__file__' in dir() else Path.cwd().parent)

from maxel.common import A179320_FP
from maxel.sequences import load_and_assign_to
from maxel.matrix import matrix_from_func, EMatrixType

## Parameters

In [2]:
n = 6  # matrix size (rows/cols)

## Load A179320 sequence

In [3]:
list_a = symbols(",".join([f"a{i}" for i in range(n)]))
dict_a = load_and_assign_to(Path(A179320_FP), list_a)

print("First", n, "terms of A179320:")
for sym, val in dict_a.items():
    print(f"  {sym} = {val}")

First 6 terms of A179320:
  a0 = 0
  a1 = 2
  a2 = -2
  a3 = 6
  a4 = -28
  a5 = 160


## Build the Lie algebra matrix (symbolic)

The lower triangular matrix $L$ with entries:
$$L[i][j] = \frac{(j+1) \cdot a_{i-j}}{(i-j)!}$$

In [4]:
def func_expr(i, j):
    return f"((j+1)*a{i-j})/factorial(i-j)"

matrix = matrix_from_func(func_expr, EMatrixType.LOWER, n)
matrix = matrix.applyfunc(factor)

print("Symbolic matrix L (log of A078812):")
pprint(matrix)

Symbolic matrix L (log of A078812):
⎡a₀    0     0     0     0     0  ⎤
⎢                                 ⎥
⎢a₁   2⋅a₀   0     0     0     0  ⎥
⎢                                 ⎥
⎢a₂                               ⎥
⎢──   2⋅a₁  3⋅a₀   0     0     0  ⎥
⎢2                                ⎥
⎢                                 ⎥
⎢a₃                               ⎥
⎢──    a₂   3⋅a₁  4⋅a₀   0     0  ⎥
⎢6                                ⎥
⎢                                 ⎥
⎢a₄    a₃   3⋅a₂                  ⎥
⎢──    ──   ────  4⋅a₁  5⋅a₀   0  ⎥
⎢24    3     2                    ⎥
⎢                                 ⎥
⎢a₅    a₄    a₃                   ⎥
⎢───   ──    ──   2⋅a₂  5⋅a₁  6⋅a₀⎥
⎣120   12    2                    ⎦


## Substitute A179320 values

In [5]:
matrix_eval = matrix.subs(dict_a)
matrix_eval = matrix_eval.applyfunc(factor)

print("Matrix L evaluated with A179320 values:")
pprint(matrix_eval)

Matrix L evaluated with A179320 values:
⎡ 0     0    0   0   0   0⎤
⎢                         ⎥
⎢ 2     0    0   0   0   0⎥
⎢                         ⎥
⎢ -1    4    0   0   0   0⎥
⎢                         ⎥
⎢ 1     -2   6   0   0   0⎥
⎢                         ⎥
⎢-7/6   2    -3  8   0   0⎥
⎢                         ⎥
⎣4/3   -7/3  3   -4  10  0⎦


## Compute exp(L) — recover A078812

Exponentiating should yield $\exp(L)[i][j] = \binom{i+j+1}{2j+1}$, the odd columns of Pascal's triangle (rectified into a lower triangular array).

In [6]:
from sympy import binomial, Matrix

exp_matrix = matrix_eval.exp()
exp_matrix = exp_matrix.applyfunc(factor)

# The full Pascal triangle needs to be 2n x 2n:
# exp(L)[i][j] = C(i+j+1, 2j+1) picks from odd columns of Pascal,
# with row index up to (n-1)+(n-1)+1 = 2n-1.
pascal_matrix = Matrix(2*n-1, 2*n-1, lambda i, j: binomial(i, j))

print("exp(L) = A078812:")
pprint(exp_matrix)
print()
print("Pascal matrix — C(i, j):")
pprint(pascal_matrix)

exp(L) = A078812:
⎡1  0   0   0   0   0⎤
⎢                    ⎥
⎢2  1   0   0   0   0⎥
⎢                    ⎥
⎢3  4   1   0   0   0⎥
⎢                    ⎥
⎢4  10  6   1   0   0⎥
⎢                    ⎥
⎢5  20  21  8   1   0⎥
⎢                    ⎥
⎣6  35  56  36  10  1⎦

Pascal matrix — C(i, j), size 2n x 2n:
⎡1  0   0    0    0    0    0    0   0   0   0⎤
⎢                                             ⎥
⎢1  1   0    0    0    0    0    0   0   0   0⎥
⎢                                             ⎥
⎢1  2   1    0    0    0    0    0   0   0   0⎥
⎢                                             ⎥
⎢1  3   3    1    0    0    0    0   0   0   0⎥
⎢                                             ⎥
⎢1  4   6    4    1    0    0    0   0   0   0⎥
⎢                                             ⎥
⎢1  5   10  10    5    1    0    0   0   0   0⎥
⎢                                             ⎥
⎢1  6   15  20   15    6    1    0   0   0   0⎥
⎢                                             ⎥
⎢1  7   21  35   